## Generate Question-Answer Pairs

We will take the chunks of text from the previous step and generate question-answer pairs based on the content of each chunk.

Each chunk can be associated to one or more questions. The questions will be generated by an LLM, but we will need to make sure that the questions are relevant to the content of the chunk.

Run this in terminal to start llama server:
```bash
./llama.cpp/llama-server.exe \
  --model models/Qwen3-4B-Q4_K_M.gguf \
  --ctx-size 8192 \
  --n-gpu-layers 999 \
  --threads 4 \
  --batch-size 2048 \
  --ubatch-size 512 \
  --flash-attn auto \
  --port 36912
```
```bash

In [18]:
import json
import re
import os
import time
import requests
from pathlib import Path
from pydantic import BaseModel
from tqdm import tqdm

# Paths
CHUNKS_PATH   = "../data/chunks/enriched_chunks.json"
OUTPUT_PATH   = "../data/qa_pairs/qa_pairs.json"
PROGRESS_PATH = "../data/qa_pairs/qa_progress.json"

LLAMA_SERVER_URL = "http://localhost:36912"

# Generation Config
MAX_TOKENS   = 2048   # increased — gives model room after thinking tokens
TEMPERATURE  = 0.7    # raised — 0.3 causes Qwen3-4B to get stuck in think loop
TOP_P        = 0.9
MAX_RETRIES  = 3
RETRY_DELAY  = 2      # small delay between retries

In [37]:
#System Prompt
system_prompt = """/no_think
## Task Description

You are generating evaluation questions for a embedding model benchmarking system that will then be used for RAG benchmarking.
You will be given content from a Maruti Suzuki Grand Vitara owner's manual.
Your job is to generate questions that a real vehicle owner would ask when using this manual.

## Critical Rule — The Most Important One

NEVER reference the chunk, the document, the manual, any diagram, or any figure in your questions.
Questions like these are STRICTLY FORBIDDEN:
- "What is mentioned in this chunk?"
- "What does the manual say about X?"
- "According to this section, what should you do?"
- "What does the diagram show?"
- "What is illustrated in the figure?"
- "What does the diagram illustrate about X?"
- "What is the man in the diagram doing?"
- "What safety symbol is included in the diagram?"

The question must stand completely alone — as if a person is asking their car's voice assistant
or a mechanic, with no knowledge of which page, section, or diagram the answer comes from.

BAD (forbidden — references diagram/chunk):
- "What does the diagram illustrate about a vehicle fire?"
- "What is the man in the diagram doing?"
- "What is the part number listed in this chunk?"
- "What warning is shown in the figure?"
- "What does the diagram show about a vehicle in water?"
- "What is the scenario illustrated in the figure?"

GOOD (required — standalone knowledge question):
- "How do I adjust the driver's seat position in the Grand Vitara?"
- "What should I do if the airbag warning light stays on?"
- "At what interval should the engine oil be changed?"
- "How do I fold the rear seats flat?"
- "Are there any service centers in Mumbai?"

## Figure Content Rules
When the content describes a diagram or figure, ONLY generate a question if the figure
contains specific actionable or factual information a user would search for independently.
While generating figure quesions, do not ask like what is mentioned in "this" or "the" figure 
because our RAG won't know which figure you are asking about in the question.

## Additional Rules

- Generate 1 to 4 questions per chunk depending on content richness.
- Ask as if the owner would ask in a natural tone during a conversation.
- Each question must be answerable SOLELY from the given chunk content.
- Questions must be specific enough that only this chunk would correctly answer them.
- You may create a realistic situation the user might be asking that question for, make it challenging and super realistic for the RAG system.
- Do NOT generate duplicate or near-duplicate questions.
- Vary question types: factual, procedural, safety, specification, figure.
- If the chunk has no meaningful vehicle-related content
  (e.g. copyright notices, part numbers, table of contents entries)
  return an empty array: []

## Output Format

Return ONLY a valid JSON array. No explanation, no markdown, no preamble.

[
  {
    "question": "A standalone realistic question a real user would ask.",
    "question_type": "factual | procedural | safety | specification | figure"
  }
]

## Examples

### Chunk (text):
The seat belt pretensioner system is designed to reduce the slack in the seat belt
during a collision. It activates automatically when the vehicle detects a frontal impact
above a certain threshold. Do not attempt to repair or modify the pretensioner system.

### Output:
[
  {
    "question": "I met with an accident yesterday and just before the impact my seat belt tightened, is it a bug or what?",
    "question_type": "factual"
  },
  {
    "question": "Can I repair the seat belt pretensioner system myself?",
    "question_type": "safety"
  }
]

### Chunk:
The engine oil must be changed every 10,000 km or 12 months, whichever comes first.
Use only SAE 5W-30 fully synthetic oil. Check oil level monthly using the dipstick.

### Output:
[
  {
    "question": "How often should I change engine oil in my Grand Vitara?",
    "question_type": "specification"
  },
  {
    "question": "What engine oil is recommended for my vehicle?",
    "question_type": "specification"
  },
  {
    "question": "I have driven my car for about 2 years now with just 7,000 kilo meters, do I need an oil change or will it work just fine as it is?",
    "question_type": "procedural"
  }
]

### Chunk:
When you adjust the image quality, perform the following procedure:
- 1) Set the parking brake firmly.
- 2) Press the engine switch to change the ignition mode to ON..
- 3) After the opening image is finished, press the camera switch (1)
- 4) 3D view mode image will be displayed. Press the button (2) for adjusting the image quality.
- 5) Adjust brightness and contrast of the image to your preference.
- 6) When the button (2) is pressed, transition from 3D view to 2D view happens on the display screen.
### Output:
[
  {
    "question": "My rear view camera's image quality is too bad, how to fix it?",
    "question_type": "procedural"
  },
  {
    "question": "I cannot see the camera display during peak sunlight hours, is there a way to increase it's brightness or visibility somehow?",
    "question_type": "procedural"
  }
]
"""

In [38]:
#Pydantic Models
from pydantic import BaseModel, field_validator
from typing import Literal

class QAPair(BaseModel):
    question: str
    question_type: Literal["factual", "procedural", "safety", "specification", "figure"] = "factual"

    @field_validator("question")
    @classmethod
    def must_not_be_empty(cls, v):
        if not v or not v.strip():
            raise ValueError("Question must not be empty")
        return v.strip()

In [ ]:
#Generation Functions
def check_server_health():
    """Check llama.cpp server is running before starting pipeline."""
    try:
        r = requests.get(f"{LLAMA_SERVER_URL}/health", timeout=5)
        if r.status_code == 200:
            print("llama.cpp server is healthy")
            return True
    except Exception:
        pass
    print("llama.cpp server is not reachable. Start it first.")
    return False


def strip_thinking_tokens(text: str) -> str:
    """
    Qwen3 emits <think>...</think> blocks before answering.
    Strip them before JSON parsing.
    """
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def extract_json_from_response(text: str) -> str:
    """
    Robustly extract JSON array from LLM response.
    Handles cases where LLM wraps JSON in markdown code blocks.
    """
    text = strip_thinking_tokens(text)

    # Try extracting from markdown code block
    match = re.search(r"```(?:json)?\s*(\[.*?\])\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    # Try finding raw JSON array
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        return match.group(0)

    return text


def parse_qa_pairs(raw_response: str) -> list[QAPair]:
    """Parse and validate LLM response into QAPair objects."""
    try:
        cleaned = extract_json_from_response(raw_response)
        data = json.loads(cleaned)
        pairs = []
        for item in data:
            try:
                pairs.append(QAPair(**item))
            except Exception as e:
                print(f"  [SKIP] Invalid QA item: {e} | item: {item}")
        return pairs
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON parse failed: {e}")
        print(f"  Raw response: '{raw_response[:300]}'")   
        return []


def get_user_prompt(chunk: dict) -> str:
    text      = chunk.get("contextualized_text") or chunk.get("text", "")
    is_figure = chunk.get("is_figure_chunk", False)

    if is_figure:
        return f"""The following is a description of a diagram or figure from a vehicle manual:

{text}

Generate practical questions a vehicle owner would ask that can be answered using this image description.
Do not refer to this image/figure/diagram directly. /no_think"""

    return f"""The following is content from a vehicle owner's manual:

{text}

Generate practical questions a vehicle owner would realistically ask a mechanic or a voice assistant that can be answered using this information. /no_think"""


def generate_qa_pairs(chunk: dict, retries: int = MAX_RETRIES) -> list[QAPair]:
    """Call llama.cpp server with retry logic."""
    url     = f"{LLAMA_SERVER_URL}/v1/chat/completions"
    headers = {"Content-Type": "application/json"}

    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": get_user_prompt(chunk)},
        ],
        "max_tokens"  : MAX_TOKENS,
        "temperature" : TEMPERATURE,
        "top_p"       : TOP_P,
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                url=url,
                headers=headers,
                json=payload,
                timeout=120    #increased timeout for larger token budget
            )
            response.raise_for_status()

            raw = response.json()

            #check finish reason — warn if truncated
            finish_reason = raw["choices"][0].get("finish_reason", "")
            if finish_reason == "length":
                print(f"  [WARN] Response truncated at max_tokens for chunk {chunk.get('chunk_id')}")

            content = raw["choices"][0]["message"]["content"].strip()

            if not content:
                print(f"  [WARN] Empty response for chunk {chunk.get('chunk_id')} — skipping")
                return []

            return parse_qa_pairs(content)

        except requests.exceptions.Timeout:
            print(f"  [TIMEOUT] Attempt {attempt}/{retries}")
        except requests.exceptions.RequestException as e:
            print(f"  [REQUEST ERROR] Attempt {attempt}/{retries}: {e}")
        except Exception as e:
            print(f"  [ERROR] Attempt {attempt}/{retries}: {e}")

        if attempt < retries:
            time.sleep(RETRY_DELAY)

    return []

In [41]:
# Load chunks
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"Total chunks: {len(all_chunks)}")

# Load progress if resuming
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

if Path(PROGRESS_PATH).exists():
    with open(PROGRESS_PATH, "r") as f:
        progress = json.load(f)
    done_chunk_ids = set(progress["done_chunk_ids"])
    qa_count       = progress.get("qa_count", 0)
    print(f"Resuming — {len(done_chunk_ids)} chunks already processed, {qa_count} QA pairs so far")
else:
    done_chunk_ids = set()
    qa_count       = 0
    print("Starting fresh")

Total chunks: 3055
Starting fresh


In [42]:
# Run Pipeline
assert check_server_health(), "Fix server before running pipeline"

SAVE_EVERY = 20

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

if Path(PROGRESS_PATH).exists():
    with open(PROGRESS_PATH, "r") as f:
        progress = json.load(f)
    done_chunk_ids = set(progress["done_chunk_ids"])
    qa_count       = progress.get("qa_count", 0)
    print(f"Resuming — {len(done_chunk_ids)} chunks done, {qa_count} QA pairs so far")
    out_file    = open(OUTPUT_PATH, "a", encoding="utf-8")
    first_entry = False
else:
    done_chunk_ids = set()
    qa_count       = 0
    print("Starting fresh")
    out_file    = open(OUTPUT_PATH, "w", encoding="utf-8")
    out_file.write("[\n")
    first_entry = True

try:
    for i, chunk in enumerate(tqdm(all_chunks, desc="Generating QA pairs")):

        chunk_id = chunk["chunk_id"]

        if chunk_id in done_chunk_ids:
            continue

        pairs = generate_qa_pairs(chunk)

        for q_idx, pair in enumerate(pairs):
            qa_entry = {
                "qa_id"              : f"{chunk_id}_q{q_idx}",
                "chunk_id"           : chunk_id,
                "page_no"            : chunk.get("page_no"),
                "question"           : pair.question,
                "question_type"      : pair.question_type,
                "chunk_text"         : chunk.get("text"),
                "contextualized_text": chunk.get("contextualized_text"),
                "heading"            : chunk.get("heading"),
                "is_figure"          : chunk.get("is_figure_chunk", False),
            }

            if not first_entry:
                out_file.write(",\n")
            json.dump(qa_entry, out_file, indent=2, ensure_ascii=False)
            out_file.flush()

            first_entry = False
            qa_count   += 1

        done_chunk_ids.add(chunk_id)

        if (i + 1) % SAVE_EVERY == 0:
            with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
                json.dump(
                    {
                        "done_chunk_ids": sorted(list(done_chunk_ids)),
                        "qa_count"      : qa_count,
                    },
                    f, indent=2
                )
            tqdm.write(f"  Checkpoint: {qa_count} QA pairs, {len(done_chunk_ids)} chunks done")

finally:
    out_file.write("\n]\n")
    out_file.close()

# Final checkpoint
with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "done_chunk_ids": sorted(list(done_chunk_ids)),
            "qa_count"      : qa_count,
        },
        f, indent=2
    )

print(f"\nTotal QA pairs generated: {qa_count}")

llama.cpp server is healthy
Starting fresh


Generating QA pairs:   1%|          | 20/3055 [01:25<3:50:07,  4.55s/it]

  Checkpoint: 65 QA pairs, 20 chunks done


Generating QA pairs:   1%|▏         | 40/3055 [02:57<3:46:56,  4.52s/it]

  Checkpoint: 135 QA pairs, 40 chunks done


...

  Checkpoint: 5919 QA pairs, 3020 chunks done


Generating QA pairs: 100%|█████████▉| 3040/3055 [3:03:24<00:37,  2.49s/it]

  Checkpoint: 5933 QA pairs, 3040 chunks done


Generating QA pairs: 100%|██████████| 3055/3055 [3:04:07<00:00,  3.62s/it]


Total QA pairs generated: 5943


In [ ]:
import json
from pathlib import Path

PROGRESS_PATH = "../data/qa_pairs/qa_progress.json"
OUTPUT_PATH   = "../data/qa_pairs/qa_pairs.json"

# Load from progress file
with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
    progress = json.load(f)

qa_pairs       = progress["qa_pairs"]
done_chunk_ids = set(progress["done_chunk_ids"])

print(f"QA pairs loaded from progress file : {len(qa_pairs)}")
print(f"Chunks completed                   : {len(done_chunk_ids)}")
print(f"\nFirst QA pair sample:")
print(json.dumps(qa_pairs[0], indent=2))
print(f"\nLast QA pair sample:")
print(json.dumps(qa_pairs[-1], indent=2))

QA pairs loaded from progress file : 10460
Chunks completed                   : 3055

First QA pair sample:
{
  "qa_id": "page_1_chunk_0_q0",
  "chunk_id": "page_1_chunk_0",
  "page_no": 1,
  "question": "What is the title of the manual mentioned in the chunk?",
  "question_type": "factual",
  "chunk_text": "N E X A\nGRAND VITARA\nOWNER'S MANUAL\nFor vehicle fitted with CNG fuel system, use Grand Vitara CNG Supplementary Owner's Manual\n\u00b7 Refer for vehicle usage at all times\n\u00b7 Contains Warranty policy and Important Information On Safety, Operation & Maintenance\nPart No. 99011M76T11-74W March, 2026 ENG",
  "contextualized_text": "N E X A\nGRAND VITARA\nOWNER'S MANUAL\nFor vehicle fitted with CNG fuel system, use Grand Vitara CNG Supplementary Owner's Manual\n\u00b7 Refer for vehicle usage at all times\n\u00b7 Contains Warranty policy and Important Information On Safety, Operation & Maintenance\nPart No. 99011M76T11-74W March, 2026 ENG",
  "heading": "N E X A",
  "is_figure":

In [ ]:
import json
from pathlib import Path
from collections import Counter

OUTPUT_PATH   = "../data/qa_pairs/qa_pairs.json"
PROGRESS_PATH = "../data/qa_pairs/qa_progress.json"

# Verify output file is valid JSON
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

print(f"Total QA pairs loaded : {len(qa_pairs)}")

# Confirm no chunk-referential questions slipped through
referential_phrases = [
    "in this chunk", "in the chunk", "in this section",
    "in this passage", "according to the chunk",
    "mentioned in this", "listed in this", "in the manual"
]
flagged = [
    p for p in qa_pairs
    if any(phrase in p["question"].lower() for phrase in referential_phrases)
]
print(f"Chunk-referential questions detected : {len(flagged)}")
if flagged:
    print("Sample flagged questions:")
    for p in flagged[:3]:
        print(f"  - {p['question']}")

# Clean up progress file
if Path(PROGRESS_PATH).exists():
    Path(PROGRESS_PATH).unlink()
    print("Progress file cleaned up")

# Summary stats
types = Counter(p["question_type"] for p in qa_pairs)
print("\nQuestion type breakdown:")
for qtype, count in types.most_common():
    print(f"  {qtype:<15} : {count}")

print(f"\nFirst entry:")
print(json.dumps(qa_pairs[0], indent=2))
print(f"\nLast entry:")
print(json.dumps(qa_pairs[-1], indent=2))

Saved 10460 QA pairs → ../data/qa_pairs/qa_pairs.json

Question type breakdown:
  specification   : 4518
  factual         : 1861
  figure          : 1854
  procedural      : 1278
  safety          : 949
